# 04 · VoiceFixer v2 — Restauración general

Notebook de inferencia con VoiceFixer v2 (paquete pip `voicefixer2`, mantenido por Render-AI-Team
a partir del VoiceFixer original de haoheliu). Categoría: **restauración general** — a diferencia
de los notebooks anteriores, este modelo intenta corregir varios tipos de degradación a la vez
(ruido, reverberación, baja resolución y clipping) dentro de un único modelo, así que aquí no hay
un baseline clásico "equivalente" único: se compara contra una cascada de los baselines clásicos
ya usados en otros notebooks (denoising + dereverb).

**Por qué `voicefixer2` y no el VoiceFixer original:** el paquete original requiere una versión
antigua de `librosa` incompatible con el `numpy` moderno de Colab, lo que rompe la instalación
out-of-the-box (un problema parecido en espíritu al de DeepFilterNet, aunque de raíz distinta:
aquí no hace falta compilar nada, es un conflicto de versiones de dependencias puramente Python).
El fork `voicefixer2` corrige justo ese conflicto y añade descarga de checkpoints vía Hugging Face,
así que la instalación es un pip normal sin parches manuales.

Requiere haber ejecutado antes `00_Setup_Base.ipynb` (Drive montado, HF_HOME configurado, utils
guardadas en `utils/`, audio del tutor subido a `audio_samples/`).


## 1. Montar Drive y configurar entorno

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/Proyecto_Audio'
os.environ['HF_HOME'] = f'{PROJECT_ROOT}/cache'
os.environ['HF_HUB_CACHE'] = f'{PROJECT_ROOT}/cache'


Mounted at /content/drive


## 2. Instalar dependencias comunes

Los paquetes de `pip` no persisten entre sesiones de Colab (solo los archivos en Drive sí), así
que hay que reinstalar estas dependencias en cada sesión nueva.


In [2]:
!pip install -q librosa soundfile scipy pesq pystoi speechmos onnxruntime matplotlib pandas


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.4 MB/s eta 0:00:00


## 3. Importar utilidades comunes (desde utils/ en Drive)

In [3]:
import sys
sys.path.append(f'{PROJECT_ROOT}/utils')

from audio_utils_funcionescomunes import cargar_audio, guardar_audio, resamplear, normalizar_pico
from audio_utils_memoria_GPU import liberar_memoria_gpu
from audio_utils_metricas_no_intrusivas import calcular_dnsmos
from audio_utils_baselines_clasicos import baseline_denoising_spectral_gating, baseline_dereverb_filtro_paso_alto


## 4. Instalar dependencias específicas de VoiceFixer v2

**El paquete `voicefixer2` ha sido retirado de PyPI** (`pip install voicefixer2` ya no encuentra
ninguna versión, aunque no siempre lo avisa a gritos si usas `-q`). La alternativa que sí funciona
es instalar directo desde el repo de GitHub que lo mantiene activamente
(`Render-AI-Team/voicefixer2`); el nombre del módulo que se importa sigue siendo `voicefixer` por
dentro, aunque el paquete/repo se llame `voicefixer2`.

Colab ya trae `ffmpeg` preinstalado, que hace falta si en algún momento procesas un archivo que no
sea `.wav` (para el audio del tutor en `.wav` no es estrictamente necesario, pero no está de más
tenerlo).


In [4]:
!pip install -q git+https://github.com/Render-AI-Team/voicefixer2


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 7.7 MB/s eta 0:00:00


## 5. Cargar el audio de prueba

Muestra los audios disponibles en `audio_samples/` y pide cuál usar.


In [5]:
carpeta_audios = f'{PROJECT_ROOT}/audio_samples'

print('Audios disponibles en audio_samples/:')
for archivo in os.listdir(carpeta_audios):
    print(f'  - {archivo}')

nombre_audio = input('\nIntroduce el nombre del archivo de audio a usar (con extensión, ej. AUDIO_TFG.wav): ').strip()
RUTA_AUDIO_ORIGINAL = f'{carpeta_audios}/{nombre_audio}'

if not os.path.isfile(RUTA_AUDIO_ORIGINAL):
    raise FileNotFoundError(f'No se ha encontrado el archivo: {RUTA_AUDIO_ORIGINAL}')

# Nombre base sin extensión, para usarlo luego al nombrar los archivos de salida
NOMBRE_BASE = os.path.splitext(nombre_audio)[0]

# Cargamos con su sample rate original, para el baseline clásico y las métricas
audio_original, sr_original = cargar_audio(RUTA_AUDIO_ORIGINAL, sr_objetivo=None, forzar_mono=True)
print(f'\nAudio cargado: {len(audio_original)/sr_original:.1f} s, {sr_original} Hz')


Audios disponibles en audio_samples/:
  - AUDIO_REVERB_ALBIOL_TFG.wav

Introduce el nombre del archivo de audio a usar (con extensión, ej. AUDIO_TFG.wav): AUDIO_REVERB_ALBIOL_TFG.wav

Audio cargado: 88.0 s, 44100 Hz


## 6. Cargar el modelo VoiceFixer v2

A diferencia de los modelos anteriores, `VoiceFixer` no expone una función de inferencia que
reciba un array de numpy: trabaja directamente con **rutas de archivo** (lee el `.wav`/`.flac` de
entrada y escribe el resultado ya en disco). El checkpoint se descarga automáticamente la primera
vez que se instancia `VoiceFixer()`.

**Los checkpoints oficiales en Hugging Face se han quedado rotos:** el paquete tiene
hard-codeada la ruta `hf://voicefixer/vocoder/model.ckpt-1490000_trimed.pt` (y también
`hf://voicefixer/voicefixer/vf.ckpt`) para descargar los pesos, pero ese repo ya no resuelve
(401 Unauthorized / repo no encontrado — parece que se renombró o se retiró). En vez de esperar a
que el mantenedor lo arregle, parcheamos la función `cached_path` que usa la librería para que,
solo para esas dos rutas concretas, descargue desde un mirror que aloja los mismos ficheros con
el mismo nombre (`Diogodiogod/voicefixer-models`, usado también por otros proyectos como TTS Audio
Suite). El resto de la librería sigue funcionando exactamente igual.

`mode=0` es el modo recomendado por defecto (modelo original). Existen también `mode=1`
(preprocesado que recorta frecuencias altas, útil si la entrada trae ruido de alta frecuencia muy
agresivo) y `mode=2` (modo "train", pensado para audio muy degradado real — justo el caso del
audio del tutor, así que merece la pena probarlo también si el modo 0 no da buen resultado).


In [6]:
import torch
from cached_path import cached_path as _cached_path_original

# Parche: redirige los dos checkpoints oficiales rotos de Hugging Face hacia un mirror que
# aloja los mismos ficheros con nombre idéntico. Si en el futuro el repo oficial vuelve a
# funcionar, este diccionario simplemente no encontrará coincidencias y no hará nada.
REDIRECCIONES_CKPT = {
    'hf://voicefixer/vocoder/model.ckpt-1490000_trimed.pt': 'hf://Diogodiogod/voicefixer-models/model.ckpt-1490000_trimed.pt',
    'hf://voicefixer/voicefixer/vf.ckpt': 'hf://Diogodiogod/voicefixer-models/vf.ckpt',
}

def _cached_path_parcheado(ruta, *args, **kwargs):
    ruta = REDIRECCIONES_CKPT.get(ruta, ruta)
    return _cached_path_original(ruta, *args, **kwargs)

import voicefixer.vocoder.config as _vf_vocoder_config
import voicefixer.base as _vf_base
_vf_vocoder_config.cached_path = _cached_path_parcheado
_vf_base.cached_path = _cached_path_parcheado

from voicefixer import VoiceFixer

usar_gpu = torch.cuda.is_available()
voicefixer_modelo = VoiceFixer()
print(f'Modelo VoiceFixer v2 cargado (checkpoints via mirror Diogodiogod/voicefixer-models). GPU disponible: {usar_gpu}')


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Modelo VoiceFixer v2 cargado (checkpoints via mirror Diogodiogod/voicefixer-models). GPU disponible: True


## 7. Inferencia sobre el audio

In [7]:
carpeta_salida_vf = f'{PROJECT_ROOT}/outputs/voicefixer'
os.makedirs(carpeta_salida_vf, exist_ok=True)
RUTA_SALIDA = f'{carpeta_salida_vf}/{NOMBRE_BASE}_restored.wav'

voicefixer_modelo.restore(
    input=RUTA_AUDIO_ORIGINAL,
    output=RUTA_SALIDA,
    cuda=usar_gpu,
    mode=0,
)
print(f'Audio restaurado guardado en: {RUTA_SALIDA}')

# VoiceFixer siempre trabaja y devuelve el audio a 44.1 kHz
audio_voicefixer, sr_voicefixer = cargar_audio(RUTA_SALIDA, sr_objetivo=None, forzar_mono=True)


100%|██████████| 3/3 [00:05<00:00,  1.99s/it]


Audio restaurado guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/voicefixer/AUDIO_REVERB_ALBIOL_TFG_restored.wav


## 8. Baseline clásico (cascada denoising + dereverb) sobre el mismo audio

Como VoiceFixer ataca varios tipos de degradación a la vez, el baseline no-IA más justo no es un
único método clásico, sino encadenar los baselines clásicos que ya usamos por separado en otros
notebooks: primero spectral gating (denoising) y después un filtro paso-alto (dereverb) sobre el
resultado. No cubre baja resolución ni clipping — para eso ya tienes baselines específicos en
`audio_utils_baselines_clasicos.py` (`baseline_bwe_interpolacion_spline`,
`baseline_declipping_interpolacion_cubica`) si más adelante quieres una comparativa más completa.


In [8]:
audio_baseline_denoise = baseline_denoising_spectral_gating(audio_original, sr_original)
audio_baseline = baseline_dereverb_filtro_paso_alto(audio_baseline_denoise, sr_original)

RUTA_BASELINE = f'{PROJECT_ROOT}/outputs/baseline_restauracion/{NOMBRE_BASE}_baseline.wav'
os.makedirs(os.path.dirname(RUTA_BASELINE), exist_ok=True)
guardar_audio(RUTA_BASELINE, audio_baseline, sr_original)
print(f'Audio baseline guardado en: {RUTA_BASELINE}')


Audio guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/baseline_restauracion/AUDIO_REVERB_ALBIOL_TFG_baseline.wav
Audio baseline guardado en: /content/drive/MyDrive/Proyecto_Audio/outputs/baseline_restauracion/AUDIO_REVERB_ALBIOL_TFG_baseline.wav


## 9. Calcular métricas (DNSMOS)

Como no hay audio limpio de referencia para la grabación del tutor, usamos DNSMOS (métrica
no-intrusiva) sobre: audio original, salida de VoiceFixer, y salida del baseline en cascada.
`calcular_dnsmos` resamplea a 16 kHz y reduce a mono automáticamente si hace falta.


In [9]:
dnsmos_original = calcular_dnsmos(audio_original, sr_original)
dnsmos_voicefixer = calcular_dnsmos(audio_voicefixer, sr_voicefixer)
dnsmos_baseline = calcular_dnsmos(audio_baseline, sr_original)

print('DNSMOS — Audio original:          ', dnsmos_original)
print('DNSMOS — VoiceFixer v2 (IA):      ', dnsmos_voicefixer)
print('DNSMOS — Baseline en cascada:     ', dnsmos_baseline)


DNSMOS — Audio original:           {'ovrl_mos': np.float64(1.3853487473266228), 'sig_mos': np.float64(1.5779884955663657), 'bak_mos': np.float64(1.790054065636629), 'p808_mos': np.float32(2.426125)}
DNSMOS — VoiceFixer v2 (IA):       {'ovrl_mos': np.float64(2.7298296690709947), 'sig_mos': np.float64(3.0837967665245354), 'bak_mos': np.float64(3.783741979532356), 'p808_mos': np.float32(2.6515336)}
DNSMOS — Baseline en cascada:      {'ovrl_mos': np.float64(1.553156446843623), 'sig_mos': np.float64(1.8252972480027962), 'bak_mos': np.float64(2.053130598184202), 'p808_mos': np.float32(2.4418974)}


## 10. Tabla resumen de la comparativa

In [10]:
import pandas as pd

resumen = pd.DataFrame([
    {'Version': 'Original (degradado)',      'OVRL': dnsmos_original.get('ovrl_mos'),   'SIG': dnsmos_original.get('sig_mos'),   'BAK': dnsmos_original.get('bak_mos')},
    {'Version': 'VoiceFixer v2 (IA)',        'OVRL': dnsmos_voicefixer.get('ovrl_mos'), 'SIG': dnsmos_voicefixer.get('sig_mos'), 'BAK': dnsmos_voicefixer.get('bak_mos')},
    {'Version': 'Baseline en cascada (no-IA)', 'OVRL': dnsmos_baseline.get('ovrl_mos'), 'SIG': dnsmos_baseline.get('sig_mos'),  'BAK': dnsmos_baseline.get('bak_mos')},
])
resumen


,Version,OVRL,SIG,BAK
0,Original (degradado),1.385349,1.577988,1.790054
1,VoiceFixer v2 (IA),2.729830,3.083797,3.783742
2,Baseline en cascada (no-IA),1.553156,1.825297,2.053131


## 11. Liberar memoria GPU

In [11]:
liberar_memoria_gpu(voicefixer_modelo, 'voicefixer_modelo')


Memoria GPU liberada. Uso actual: 0.01 GB


## Próximo paso

Con VoiceFixer v2 ya probado y comparado contra su baseline en cascada, el siguiente notebook
sería AudioSR (super-resolución) o ClearVoice/MossFormer2, según el orden que fijemos, siguiendo
la Fase 2 del proyecto.
